# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [8]:
from bs4 import BeautifulSoup

file = './rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [9]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [10]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [11]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [12]:
import re

# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter only links that point to actual allrecipes.com recipe pages
# (the naive "recipe" in href filter also matches nav/category/login links)
RECIPE_URL_RE = re.compile(r"allrecipes\.com/recipe/\d+/")

recipe_urls = sorted({
    link["href"] for link in recipe_links
    if RECIPE_URL_RE.search(link["href"])
})

print(f"Linked Recipes ({len(recipe_urls)}):")
for url in recipe_urls:
    print(url)

Linked Recipes (16):
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/214618/beer-can-chicken/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque

In [13]:
# The local HTML file is itself a recipe page (Rotisserie Chicken).
# We already have it parsed in `soup`, so it becomes the first document
# of the corpus without needing to be downloaded again.
LOCAL_RECIPE_URL = "https://www.allrecipes.com/recipe/93168/rotisserie-chicken/"

all_recipe_urls = [LOCAL_RECIPE_URL] + recipe_urls
print(f"Total recipes to build the corpus: {len(all_recipe_urls)}")

Total recipes to build the corpus: 17


### Crawler: descargar y parsear cada receta enlazada

Para completar el corpus necesitamos visitar cada una de las URLs encontradas y
aplicar sobre ellas la misma extracción que hicimos en la Parte 2 sobre el HTML
local. Se define:

1. `fetch_html(url)`: descarga el HTML con un User-Agent de navegador (allrecipes
   bloquea clientes sin `User-Agent`) y una pequeña pausa entre requests para no
   saturar el servidor.
2. `parse_recipe(html, url)`: reutiliza los selectores de la Parte 2 para devolver
   un diccionario con título, descripción, ingredientes, instrucciones y
   nutrición.
3. Un loop que aplica ambas funciones sobre todas las `all_recipe_urls`, salta la
   descarga para la receta local (ya la tenemos parseada en `soup`) y guarda el
   resultado en `recipes_corpus.json`.

In [14]:
import time
import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def fetch_html(url: str, timeout: int = 15) -> str | None:
    """Download the raw HTML for a recipe URL. Returns None on failure."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"  [ERROR] {url} -> {e}")
        return None

In [15]:
def parse_recipe(page_soup: BeautifulSoup, url: str) -> dict | None:
    """Extract title/description/ingredients/instructions/nutrition from a
    recipe page, reusing the same selectors validated in Parte 2.
    Returns None if the page doesn't look like a recipe (missing title)."""

    title_tag = page_soup.find("meta", {"property": "og:title"})
    if title_tag is None:
        return None
    title = title_tag["content"]

    description_tag = page_soup.find("meta", {"name": "description"})
    description = description_tag["content"] if description_tag else ""

    ingredients_section = page_soup.find_all(
        "li", class_="mm-recipes-structured-ingredients__list-item"
    )
    ingredients = [i.get_text().strip() for i in ingredients_section]

    instructions_section = page_soup.find_all(
        "p", class_="comp mntl-sc-block mntl-sc-block-html"
    )
    instructions = [i.get_text().strip() for i in instructions_section]

    nutrition_section = page_soup.find_all(
        "span",
        class_="mm-recipes-nutrition-facts-label__nutrient-name "
        "mm-recipes-nutrition-facts-label__nutrient-name--has-postfix",
    )
    nutrition_facts = [
        fact.parent.get_text().strip().replace("\n", " ") for fact in nutrition_section
    ]

    # Calories live in a separate row (no "--has-postfix" span, since it has
    # no unit suffix), so the selector above misses it. Add it explicitly.
    calories_row = page_soup.find(
        "tr", class_="mm-recipes-nutrition-facts-label__calories"
    )
    if calories_row is not None:
        calories_text = calories_row.get_text().strip().replace("\n", " ")
        nutrition_facts = [calories_text] + nutrition_facts

    return {
        "url": url,
        "title": title,
        "description": description,
        "ingredients": ingredients,
        "instructions": instructions,
        "nutrition": nutrition_facts,
    }

In [16]:
import json

corpus = []
failed_urls = []

for url in all_recipe_urls:
    if url == LOCAL_RECIPE_URL:
        # Already parsed from the local file, no need to download it again
        page_soup = soup
    else:
        html = fetch_html(url)
        time.sleep(1)  # be polite with the server
        if html is None:
            failed_urls.append(url)
            continue
        page_soup = BeautifulSoup(html, "html.parser")

    recipe = parse_recipe(page_soup, url)
    if recipe is None:
        failed_urls.append(url)
        continue

    corpus.append(recipe)
    print(f"OK  - {recipe['title']}")

print()
print(f"Recetas parseadas correctamente: {len(corpus)}")
print(f"Recetas fallidas: {len(failed_urls)}")
for url in failed_urls:
    print("  -", url)

with open("recipes_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print("\nCorpus guardado en recipes_corpus.json")

OK  - Rotisserie Chicken
OK  - Beer Butt Chicken
OK  - Drunk Chicken
OK  - Beer Can Chicken
OK  - Best Beer Can Chicken
OK  - Good Frickin’ Paprika Chicken
OK  - Smoked Beer Butt Chicken
OK  - The Best Beer Can Chicken Ever
OK  - Cilantro-Lime Grilled Chicken
OK  - Rosemary Buttermilk Chicken
OK  - Miso Honey Chicken
OK  - Grilled Spatchcocked Chicken
OK  - Grilled Chicken Under a Brick
OK  - Buttermilk Barbecue Chicken
OK  - Smoked Whole Chicken
OK  - Easy Barbeque Chicken
OK  - Darn Good Chicken

Recetas parseadas correctamente: 17
Recetas fallidas: 0

Corpus guardado en recipes_corpus.json


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [17]:
%pip install -q sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


### Paso 1: preparar los documentos (chunking)

Por cada receta armamos un texto plano con título, descripción, ingredientes,
instrucciones y datos de nutrición, y lo dividimos en fragmentos (`chunks`) de
tamaño acotado. Esto es necesario porque el modelo de embeddings trunca el
texto a ~512 tokens: una receta completa (con sus instrucciones e historias)
supera ese límite, así que sin chunking perderíamos silenciosamente todo el
contenido que exceda el corte. Cada chunk conserva metadata (`title`, `url`)
para poder rastrear su origen.

In [18]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100) -> list[str]:
    """Character-based chunking with overlap, so ideas aren't cut in half."""
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = end - overlap
    return chunks


def recipe_to_text(recipe: dict) -> str:
    parts = [
        recipe["title"],
        recipe["description"],
        "Ingredients: " + "; ".join(recipe["ingredients"]),
        "Instructions: " + " ".join(recipe["instructions"]),
        "Nutrition facts: " + "; ".join(recipe["nutrition"]),
    ]
    return "\n".join(p for p in parts if p)


# Load the corpus built by the crawler
with open("recipes_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

documents = []   # chunk text
metadatas = []   # {title, url}
ids = []         # unique chunk id

for recipe in corpus:
    full_text = recipe_to_text(recipe)
    for i, chunk in enumerate(chunk_text(full_text)):
        documents.append(chunk)
        metadatas.append({"title": recipe["title"], "url": recipe["url"]})
        ids.append(f"{recipe['url']}::chunk{i}")

print(f"Recetas: {len(corpus)}  ->  Chunks generados: {len(documents)}")

Recetas: 17  ->  Chunks generados: 52


### Paso 2: generar embeddings

Usamos `intfloat/e5-small-v2` (mismo modelo usado en el ejercicio de bases de
datos vectoriales). E5 requiere anteponer `"passage: "` al indexar documentos y
`"query: "` al buscar, y trabajamos con embeddings normalizados para que el
producto interno equivalga a similitud coseno.

In [19]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "intfloat/e5-small-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(MODEL_NAME, device=device)
print("Dispositivo:", device, "| dim:", embed_model.get_sentence_embedding_dimension())

passages = ["passage: " + d for d in documents]
chunk_embeddings = embed_model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print(chunk_embeddings.shape)


def embed_query(query: str):
    vec = embed_model.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    return vec

c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4755.01it/s]
C:\Users\david\AppData\Local\Temp\ipykernel_34604\3683408328.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dispositivo:", device, "| dim:", embed_model.get_sentence_embedding_dimension())


Dispositivo: cpu | dim: 384


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.14it/s]

(52, 384)


### Paso 3: indexar en ChromaDB

ChromaDB en memoria (sin servidor) es la opción de menor fricción para
prototipar, igual que en el ejercicio de bases de datos vectoriales.

In [20]:
import chromadb

chroma_client = chromadb.Client()
COLLECTION_NAME = "recipes"

if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=documents,
    metadatas=metadatas,
)

print("Chunks indexados en Chroma:", collection.count())

Chunks indexados en Chroma: 52


### Paso 4: retrieval — función `search`

Dada una consulta en lenguaje natural, la codificamos con `embed_query` y
pedimos a Chroma los `k` chunks más similares (coseno).

In [21]:
def search(query: str, k: int = 5) -> list[dict]:
    """Return the top-k chunks most relevant to the query."""
    query_vec = embed_query(query)
    results = collection.query(query_embeddings=query_vec.tolist(), n_results=k)

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append({
            "text": doc,
            "title": meta["title"],
            "url": meta["url"],
            "score": 1 - dist,  # cosine distance -> similarity
        })
    return hits


# Quick sanity check
for hit in search("beer can chicken", k=3):
    print(f"[{hit['score']:.3f}] {hit['title']}")
    print("   ", hit["text"][:120].replace("\n", " "), "...")

[0.913] Best Beer Can Chicken
    Best Beer Can Chicken In this best beer can chicken recipe, chicken is seasoned with a sweet and spicy brown sugar-papri ...
[0.900] Beer Can Chicken
    Beer Can Chicken This beer can chicken cooks a perfectly seasoned whole chicken on a grill until it's deliciously crisp  ...
[0.893] Beer Butt Chicken
    Beer Butt Chicken This beer butter chicken recipe combines beer, butter, chicken, and seasonings for a moist and flavorf ...


### Paso 5: generación — función `rag`

Se arma un prompt con los chunks recuperados como contexto y se envía a un LLM.

In [22]:
%pip install -q google-generativeai

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.


In [ ]:
import os
from google import genai

client = genai.Client(api_key='API_KEY_HERE')  # Replace with your actual API key


def build_prompt(query: str, hits: list[dict]) -> str:
    context = "\n\n".join(
        f"[{h['title']}] {h['text']}" for h in hits
    )
    return (
        "Answer the question using only the context below. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )


def rag(query: str, k: int = 3) -> str:
    hits = search(query, k=k)
    prompt = build_prompt(query, hits)
    response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=prompt,
    )
    answer = response.text
    print(answer)
    print("=" * 80)
    print(hits)
    return answer

### Demo: consultas de ejemplo

In [34]:
demo_queries = [
    "What recipies can I prepare with chiken?",
    "What recipes use beer to cook chicken?",
    "How many calories does rotisserie chicken have?",
]

for q in demo_queries:
    print("=" * 80)
    print("QUERY:", q)
    print("-" * 80)
    rag(q)
    print()

QUERY: What recipies can I prepare with chiken?
--------------------------------------------------------------------------------
Based on the provided context, the recipes you can prepare with chicken are:

* Best Beer Can Chicken
* Grilled Spatchcocked Chicken
* Smoked Whole Chicken
[{'text': "into each can. Be careful, this will make beer foam up and out of the can. Rub each chicken with 2 tablespoons oil. Rub remaining seasoning mix over entire chickens, inside and out. Place each chicken upright over 1 beer can. Drain wood chips; place with coals, in an aluminum pan, or under the grill grate according to manufacturer's instructions. Place cans with chicken directly on the grill. Close the lid and grill chicken until no longer pink, and juices run clear, about 1 ½ hours. An instant-read thermometer inserted into the thickest part of thighs should read at least 165 degrees F (74 degrees C). Remove chickens from the grill; discard the beer cans. Cover chickens with a doubled sheet of 